In [1]:
import pandas as pd              # Para análisis de datos
import numpy as np               # Para cálculos numéricos
from scipy import linalg 

Este método es uno iterativo que funciona de manera similar al método de punto fijo para funciones. La idea es que si tenemos $A\vec{x}=\vec{b}$, y reescribiendo $A = M + (A-M)$, podemos depejar y obtenemos $M\vec{x} = (M-A)\vec{x}+\vec{b}$. De esta forma el método iterativo queda como $M\vec{x}^{(k)} = (M-A)\vec{x}^{(k-1)}+\vec{b}$. 

El método de Jacobi (método DLU) consiste en descomponer $A = D - L - U$, siendo $D$ una matriz diagonal, $-L$ la matriz triangular inferior con diagonal de ceros, y $-U$ la matriz triangular superior con diagonal de ceros. De esta forma $M=D$ y $M-A = L + U$. Por lo tanto:

$\vec{x}^{(k)} = D^{-1} (L+U) \vec{x}^{(k-1)} + D^{-1}\vec{b} = T_{J}\vec{x}^{(k-1)} + \vec{c}_{J}$. 

La incertidumbre de iteración (según chatGPT) es $err^{(k)} \approx \frac{||\vec{x}^{(k)}-\vec{x}^{(k-1)}||}{1-||T_{J}||}$, y por tanto, para que el método converja, $||T_{J}|| < 1$. 

In [2]:
def descomp_DLU(A,dim):
    # A = matriz a descomponer en DLU
    # dim = dimension de la matriz cuadrada A

    D = np.zeros((dim,dim))
    L = np.zeros((dim,dim))
    U = np.zeros((dim,dim))

    for k in range (0,dim,1): # Matriz diagonal
        D[k][k] = A[k][k]

    for k in range (0,dim-1,1):
        for m in range (k+1,dim,1):
            L[m][k] = -A[m][k]

    for k in range (1,dim,1):
        for m in range (k-1,-1,-1):
            U[m][k] = -A[m][k]

    return D,L,U

In [3]:
def proced_DLU(A,b,x0,err,maxit):
    # A = matriz a descomponer en DLU
    # b = vector b tal que Ax=b
    # x0 = vector valor inicial de busqueda
    # err = incertidumbre estimada
    # maxit = numero maximo de iteraciones 

    dimfil,dimcol = A.shape

    if dimfil == dimcol:
        dim = dimfil

        D,L,U = descomp_DLU(A,dim)

        cj = np.dot(np.linalg.inv(D),b)
        Tj = np.dot(np.linalg.inv(D),(L+U))

        Tj_norm_inf = np.linalg.norm(Tj,np.inf) 

        if Tj_norm_inf >= 1:
            print('El procedimiento no converge.')
            return
        else:

            iter = 0
            errest = 10000 # por poner algo 

            x = np.dot(Tj,x0) + cj

            while maxit > iter and errest > err:
                x0 = x
                x = np.dot(Tj,x0) + cj
                iter = iter + 1
                errest = (np.linalg.norm(x-x0,np.inf))/(1-Tj_norm_inf)


            return x,iter,errest


    else:
        print('Error: A no es cuadrada.')
        return

Ejemplo:

$A\vec{x} = \begin{pmatrix} 
4 & -1 & 1 \\ 
4 & -8 & 1 \\
-2 & 1 & 5
\end{pmatrix} 
\vec{x} =
\begin{pmatrix} 
7 \\ 
-21 \\
15 
\end{pmatrix} 
= \vec{b} $



In [4]:
A = np.array([[4,-1,1],[4,-8,1],[-2,1,5]])

#dim,dim2 = A.shape

#D,L,U = descomp_DLU(A,dim)

#print(D)
#print(L)
#print(U)
#print(D-L-U)

b = np.array([[7],[-21],[15]])
x0 = np.array([[1],[2],[2]])
maxit = 5
err = 10^(-6)

sol,iter,err = proced_DLU(A,b,x0,err,maxit)

print("La solucion es x =",sol,'.')
print('El procedimiento se ha realizado en',iter,'iteraciones y con un error de',err,'.')

La solucion es x = [[1.99859375]
 [3.9971875 ]
 [2.99859375]] .
El procedimiento se ha realizado en 5 iteraciones y con un error de 0.011874999999999858 .
